# 대출 데이터 전처리
원본: `csv/raw 데이터/loans_raw.csv`. 대출 한 건당 한 행이다.
주택 보유 고객의 대출만 포함하므로 전체 고객의 대출로 해석하지 않는다.
원본 22개 컬럼과 모든 행을 보존하고 날짜·참조 ID의 자료형만 정리한다.
이상 사례는 삭제·상한 처리하지 않고 검토 목록으로 저장한다.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# 프로젝트 루트 또는 하위 폴더에서 실행 가능
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'csv/raw 데이터/loans_raw.csv').exists())
source = root / 'csv/raw 데이터/loans_raw.csv'
loan_df = pd.read_csv(source)
raw_df = loan_df.copy()
print(f'원본: {len(loan_df):,}행 × {loan_df.shape[1]}개 컬럼')
display(loan_df.head())


## 0. 원본 대비 구성·컬럼·행·값 검증
`full data/loans_202610061807.csv`와 전처리 전 파일을 비교한다.
다른 테이블의 컬럼이 추가되었는지, 원본 컬럼이 빠졌는지, 고객 필터로 설명되는 행 집합인지 확인한다.
최초 SQL·생성 코드가 없으므로 실제 join 실행 여부는 확정할 수 없다.
컬럼·ID·전체 값이 일치하면 원본 대출의 고객 필터 결과와 동일한 데이터라고 판단한다.


In [ ]:
original = pd.read_csv(root / 'full data/loans_202610061807.csv')
customers = pd.read_csv(root / 'csv/raw 데이터/customers.csv', usecols=['customer_id'])
assert original.loan_id.is_unique and raw_df.loan_id.is_unique
removed_cols = sorted(set(original.columns) - set(raw_df.columns))
added_cols = sorted(set(raw_df.columns) - set(original.columns))
print(f'full data 원본: {len(original):,}행 × {original.shape[1]}컬럼')
print(f'raw 대출 파일: {len(raw_df):,}행 × {raw_df.shape[1]}컬럼')
print('제거된 컬럼:', removed_cols)
print('추가된 컬럼:', added_cols)
print('컬럼 순서 동일:', list(original.columns) == list(raw_df.columns))
original_ids = set(original.loan_id)
raw_ids = set(raw_df.loan_id)
print('원본에 없는 대출 ID:', len(raw_ids - original_ids))
excluded = original.loc[~original.loan_id.isin(raw_ids)]
print('제외된 대출 행:', len(excluded))
display(excluded.loan_type.value_counts().rename('제외 건수').to_frame())
expected = original.loc[original.customer_id.isin(customers.customer_id)]
print('customers 고객 필터와 ID 집합 일치:', set(expected.loan_id) == raw_ids)
print('제외 행이 모두 customers 밖의 고객:', (~excluded.customer_id.isin(customers.customer_id)).all())

common = sorted(original_ids & raw_ids)
left = original.set_index('loan_id').loc[common]
right = raw_df.set_index('loan_id').loc[common]
differences = {}
for col in original.columns.drop('loan_id'):
    if col not in right.columns:
        continue
    a, b = left[col], right[col]
    if col in ['opened_at', 'maturity_date']:
        a, b = pd.to_datetime(a, errors='raise'), pd.to_datetime(b, errors='raise')
        same = a.eq(b) | (a.isna() & b.isna())
    elif pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        same = np.isclose(a, b, rtol=1e-10, atol=1e-6, equal_nan=True)
    else:
        same = a.eq(b) | (a.isna() & b.isna())
    differences[col] = int((~same).sum())
display(pd.Series(differences, name='값 불일치 건수').to_frame())
assert not removed_cols and not added_cols
assert raw_ids == set(expected.loan_id)
assert sum(differences.values()) == 0
print('결론: 원본 대출 22컬럼을 그대로 유지한 customers 고객 필터 결과와 일치.')
print('다른 테이블의 속성 컬럼 추가나 원본 컬럼 제거는 없음. 최초 결합 방식은 생성 코드 없이 확정 불가.')


## 1. 결측·중복·대출 유형 확인
고객 ID 중복은 여러 대출을 보유한 정상적인 경우다. 대출 ID 중복과 구분한다.

In [ ]:
check_df = pd.DataFrame({
    '자료형': loan_df.dtypes.astype(str),
    '결측 개수': loan_df.isna().sum(),
    '0 개수': loan_df.eq(0).sum(),
    '고유값 수': loan_df.nunique(),
})
display(check_df)
print('전체 중복 행:', loan_df.duplicated().sum())
print('대출 ID 중복:', loan_df.loan_id.duplicated().sum())
print('대출 보유 고객 수:', loan_df.customer_id.nunique())
display(pd.crosstab(loan_df.loan_type, loan_df.loan_purpose))
display(pd.crosstab(loan_df.loan_type, loan_df.collateral_type))


## 2. 날짜·참조 ID 자료형 정리
날짜 변환 실패와 소수 ID는 오류로 표시한다. 비담보대출의 LTV 0은 원본 그대로 보존하고 유형별로 해석한다.

In [ ]:
for col in ['opened_at', 'maturity_date']:
    loan_df[col] = pd.to_datetime(loan_df[col], errors='raise')
assert loan_df.reference_real_asset_id.dropna().mod(1).eq(0).all(), '소수 부동산 ID 확인 필요'
loan_df['reference_real_asset_id'] = loan_df.reference_real_asset_id.astype('Int64')
display(loan_df[['opened_at', 'maturity_date']].agg(['min', 'max']))
display(loan_df.groupby('loan_type')[['interest_rate', 'term_years',
    'original_principal', 'outstanding_balance', 'current_ltv']].agg(['min', 'median', 'max']))


## 3. 금액·상환액·연결 관계 검증
- `original_principal`: 최초 대출금. `outstanding_balance`: 현재 잔액.
- `principal`과 현재 잔액이 같은지 확인하되 컬럼을 임의 삭제하지 않는다.
- 연 상환액과 월 상환액 × 12는 원 단위 반올림을 고려해 0.1원까지 허용한다.
- LTV 1 초과는 검토 대상이며 오류라고 단정하지 않는다.
- 날짜 검토 기준은 기존 데이터 기준월 말일인 2026-09-30이다.


In [ ]:
reference_date = pd.Timestamp('2026-09-30')
amount_cols = ['original_principal', 'principal', 'outstanding_balance',
               'monthly_payment_estimated', 'annual_debt_service']
issues = pd.DataFrame(index=loan_df.index)
issues['금액_음수'] = loan_df[amount_cols].lt(0).any(axis=1)
issues['금액_비유한값'] = ~np.isfinite(loan_df[amount_cols]).all(axis=1)
issues['금리_0이하'] = loan_df.interest_rate.le(0)
issues['기간_0이하'] = loan_df.term_years.le(0)
issues['잔액_최초원금초과'] = loan_df.outstanding_balance.gt(loan_df.original_principal + .01)
issues['principal_잔액불일치'] = ~np.isclose(loan_df.principal, loan_df.outstanding_balance, rtol=0, atol=.01)
issues['연월상환액불일치'] = ~np.isclose(loan_df.annual_debt_service,
    loan_df.monthly_payment_estimated * 12, rtol=0, atol=.1)
issues['만기_개설일이전'] = loan_df.maturity_date.lt(loan_df.opened_at)
issues['기준일이후개설'] = loan_df.opened_at.gt(reference_date)
issues['기준일이전만기_ACTIVE'] = loan_df.maturity_date.lt(reference_date) & loan_df.status.eq('ACTIVE')
issues['현재LTV_1초과'] = loan_df.current_ltv.gt(1)
issues['LTV_음수'] = loan_df[['ltv_at_origination', 'current_ltv']].lt(0).any(axis=1)
issues['연체일수_음수'] = loan_df.overdue_days.lt(0)
for col in ['housing_purchase_linked', 'is_credit_line', 'policy_cap_applied']:
    issues[col + '_이진값오류'] = ~loan_df[col].isin([0, 1])
issues['신용한도유형_플래그불일치'] = loan_df.loan_type.eq('CREDIT_LINE').ne(loan_df.is_credit_line.eq(1))

assets = pd.read_csv(root / 'csv/raw 데이터/real_estate_raw.csv',
                     usecols=['real_asset_id', 'customer_id'])
assert assets.real_asset_id.is_unique
owner_map = assets.set_index('real_asset_id').customer_id
owners = loan_df.reference_real_asset_id.map(owner_map)
issues['참조주택없음'] = owners.isna()
issues['참조주택_고객불일치'] = owners.notna() & owners.ne(loan_df.customer_id)
display(issues.sum().rename('검토 건수').to_frame())
review_df = pd.concat([loan_df, issues], axis=1).loc[issues.any(axis=1)].copy()
display(review_df.head(10))


## 4. 주택 매입 파일과 대출 집계 대조
매입에 연결된 대출만 주택 ID로 집계한다. 개인 생활비·사업 대출은 제외한다.
원본 매입 파일과 대조하므로 매입 전처리에서 행을 제외했더라도 독립적으로 검증할 수 있다.
대출 없는 주택은 합계가 0이다. 원본 대출 행을 매입 파일에 바로 결합하면 행이 늘어난다.


In [ ]:
home = pd.read_csv(root / 'csv/raw 데이터/home_purchases_loanremove.csv')
assert home.real_asset_id.is_unique
linked = loan_df.loc[loan_df.housing_purchase_linked.eq(1)]
purchase_sum = linked.groupby('reference_real_asset_id').agg(
    total_loan_principal=('original_principal', 'sum'),
    monthly_payment_total=('monthly_payment_estimated', 'sum'),
    annual_debt_service_total=('annual_debt_service', 'sum'))
comparison = home[['real_asset_id', *purchase_sum.columns]].merge(
    purchase_sum, left_on='real_asset_id', right_index=True,
    how='left', validate='one_to_one', suffixes=('_home', '_loans'))
for col in purchase_sum.columns:
    a = comparison[col + '_home'].fillna(0)
    b = comparison[col + '_loans'].fillna(0)
    print(col, '불일치:', int((~np.isclose(a, b, rtol=1e-10, atol=.1)).sum()))


1. 만기가 지났는데 ACTIVE인 13,211건
- 대출 유형별 건수를 확인
- opened_at에 term_years를 더한 날짜가 maturity_date와 맞는지 확인
- 갱신 기록이 없다면 상태 오류인지 갱신 미반영인지 확정하기 어려우므로 우선 유지

In [ ]:
# 1. 기준일보다 만기가 지났는데 ACTIVE인 대출
reference_date = pd.Timestamp("2026-09-30")

expired_active = loan_df.loc[
    (loan_df["maturity_date"] < reference_date)
    & (loan_df["status"] == "ACTIVE")
].copy()

print(f"만기 경과 ACTIVE 대출: {len(expired_active):,}건")

# 대출 유형별 건수
display(
    expired_active["loan_type"]
    .value_counts()
    .rename_axis("대출 유형")
    .to_frame("건수")
)

# 2. 개설일 + 대출기간으로 예상 만기일 계산
expired_active["예상_만기일"] = expired_active.apply(
    lambda row: row["opened_at"]
    + pd.DateOffset(years=int(row["term_years"])),
    axis=1
)

expired_active["만기일_계산일치"] = (
    expired_active["maturity_date"]
    == expired_active["예상_만기일"]
)

# 유형별 계산 일치·불일치 건수
display(
    pd.crosstab(
        expired_active["loan_type"],
        expired_active["만기일_계산일치"]
    )
    .reindex(columns=[False, True], fill_value=0)
    .rename(columns={False: "계산 불일치", True: "계산 일치"})
)

# 3. 실제 사례 확인
display(
    expired_active[[
        "loan_id",
        "loan_type",
        "opened_at",
        "term_years",
        "maturity_date",
        "예상_만기일",
        "만기일_계산일치",
        "status",
        "outstanding_balance",
        "overdue_days",
    ]].head(20)
)

만기 경과 ACTIVE 대출 13,211건은 모두 CREDIT_LINE이며, 만기일은 개설일에 약정기간 1년을 더한 날짜와 일치한다. 날짜 계산 불일치는 없으나, 갱신 이력이 없어 현재 상태와의 관계는 확인할 수 없다. 해당 행과 원본 값은 유지하고, 현재 만기 여부 판단에는 사용을 보류한다.

2. ltv 1 초과

In [ ]:
# 1. 현재 LTV가 100%를 초과하는 대출
ltv_check = loan_df.loc[loan_df["current_ltv"] > 1].copy()

print(f"현재 LTV 100% 초과: {len(ltv_check):,}건")
display(ltv_check["loan_type"].value_counts().to_frame("건수"))

# 2. 연결된 주택의 평가금액 가져오기
assets = pd.read_csv(
    root / "csv/raw 데이터/real_estate_raw.csv",
    usecols=["real_asset_id", "estimated_value"]
)

ltv_check = ltv_check.merge(
    assets,
    left_on="reference_real_asset_id",
    right_on="real_asset_id",
    how="left",
    validate="many_to_one"
)

print("평가금액 연결 실패:", ltv_check["estimated_value"].isna().sum())
print("평가금액 0 이하:", ltv_check["estimated_value"].le(0).sum())

# 3. 잔액 ÷ 평가금액으로 LTV 재계산
valid_value = ltv_check["estimated_value"].gt(0)

ltv_check["재계산_LTV"] = (
    ltv_check["outstanding_balance"]
    / ltv_check["estimated_value"].where(valid_value)
)

# 원본 LTV의 소수점 반올림을 고려해 비교
ltv_check["계산일치"] = np.isclose(
    ltv_check["current_ltv"],
    ltv_check["재계산_LTV"],
    rtol=0,
    atol=0.00001,
    equal_nan=False
)

display(
    ltv_check["계산일치"]
    .value_counts()
    .rename_axis("계산 일치 여부")
    .to_frame("건수")
)

display(
    ltv_check[[
        "loan_id",
        "customer_id",
        "loan_type",
        "reference_real_asset_id",
        "outstanding_balance",
        "estimated_value",
        "current_ltv",
        "재계산_LTV",
        "계산일치"
    ]].head(20)
)

In [ ]:
# 1. 대출 유형별 LTV의 0 개수
ltv_zero_check = loan_df.groupby("loan_type").agg(
    전체건수=("loan_id", "size"),
    최초LTV_0건수=("ltv_at_origination", lambda s: s.eq(0).sum()),
    현재LTV_0건수=("current_ltv", lambda s: s.eq(0).sum()),
)

display(ltv_zero_check)

# 담보 여부와 LTV 0의 관계
display(
    pd.crosstab(
        loan_df["collateral_type"],
        loan_df["current_ltv"].eq(0)
    )
    .reindex(columns=[False, True], fill_value=0)
    .rename(columns={False: "현재LTV 0 아님", True: "현재LTV 0"})
)

# 2. 0/1 플래그의 값 분포
flag_cols = [
    "housing_purchase_linked",
    "is_credit_line",
    "policy_cap_applied",
]

for col in flag_cols:
    print(f"\n[{col}]")
    display(
        loan_df[col]
        .value_counts(dropna=False)
        .sort_index()
        .to_frame("건수")
    )

# 3. 대출 유형별 연체 여부
display(
    pd.crosstab(
        loan_df["loan_type"],
        loan_df["overdue_days"].gt(0)
    )
    .reindex(columns=[False, True], fill_value=0)
    .rename(columns={False: "연체일수 0 이하", True: "연체일수 양수"})
)

print("연체일수 결측:", loan_df["overdue_days"].isna().sum())
print("연체일수 음수:", loan_df["overdue_days"].lt(0).sum())

1. 연체일수 377 은 모두 개인대출.
2. 비담보대출의 LTV 0은 ‘비해당’, 이진 플래그의 0은 ‘아니오’, 연체일수 0은 ‘연체 없음’을 의미하므로 그대로 유지한다.
3. LTV 통계는 주택담보대출만 대상으로 계산하며, 연체일수가 양수인 개인대출 377건도 분석에 필요한 정보로 보존한다.
4. 만기 경과 ACTIVE 13,211건과 LTV 100% 초과 443건은 오류로 확정할 근거가 없어 수정·삭제하지 않는다.

In [ ]:
loan_df.to_csv(
    root / "csv/re전처리/loans_전처리.csv",
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

print("loans_전처리.csv 저장 완료")